# 2 预备知识

## 2.1 数据操作

首先，我们介绍$n$维数组，也称为*张量*（tensor）。
使用过Python中NumPy计算包的读者会对本部分很熟悉。
无论使用哪个深度学习框架，它的*张量类*（在MXNet中为`ndarray`，
在PyTorch和TensorFlow中为`Tensor`）都与Numpy的`ndarray`类似。
但深度学习框架又比Numpy的`ndarray`多一些重要功能：
首先，GPU很好地支持加速计算，而NumPy仅支持CPU计算；
其次，张量类支持自动微分。
这些功能使得张量类更适合深度学习。
如果没有特殊说明，本书中所说的张量均指的是张量类的实例。

### 2.1.1 入门



In [1]:
# 环境里有两份 libiomp5md.dll（torch 自带的一份 + conda 的一份），
# 画图时第二次初始化 OpenMP 会直接 abort 掉内核，所以这行要放在最前面。
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"


In [2]:
import torch
x=torch.arange(12)
x

tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11])

张量表示一个由数值组成的数组，这个数组可能有多个维度。 具有一个轴的张量对应数学上的向量（vector）； 具有两个轴的张量对应数学上的矩阵（matrix）； 具有两个轴以上的张量没有特殊的数学名称。

首先，我们可以使用 arange 创建一个行向量 x。这个行向量包含以0开始的前12个整数，它们默认创建为整数。也可指定创建类型为浮点数。张量中的每个值都称为张量的 元素（element）。例如，张量 x 中有 12 个元素。除非额外指定，新的张量将存储在内存中，并采用基于CPU的计算。

In [3]:
x.shape # 可以通过张量的shape属性来访问张量（沿每个轴的长度）的形状 。


torch.Size([12])

In [4]:
x.numel() # 张量的元素个数可以通过numel函数返回

12

In [5]:
X=x.reshape(3,4) # 通过调用reshape函数来改变张量的形状。新形状必须与原始形状具有相同的元素个数。
X

tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])

In [6]:
X=x.reshape(-1,4) # 通过在一个维度上使用-1来推断该维度的大小。
X

tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])

In [7]:
torch.zeros((2, 3, 4)) #我们希望使用全0、全1、其他常量，或者从特定分布中随机采样的数字来初始化矩阵。

tensor([[[0., 0., 0., 0.],
         [0., 0., 0., 0.],
         [0., 0., 0., 0.]],

        [[0., 0., 0., 0.],
         [0., 0., 0., 0.],
         [0., 0., 0., 0.]]])

In [8]:
torch.ones((2, 3, 4)) #全为1的张量


tensor([[[1., 1., 1., 1.],
         [1., 1., 1., 1.],
         [1., 1., 1., 1.]],

        [[1., 1., 1., 1.],
         [1., 1., 1., 1.],
         [1., 1., 1., 1.]]])

In [9]:
torch.randn(3, 4) #从标准正态分布中随机采样的张量


tensor([[ 0.6000, -0.7934, -0.0471, -0.8659],
        [ 1.4295,  1.0628,  2.0344, -0.0214],
        [ 0.2810,  0.4662, -1.2403, -0.1011]])

In [10]:
torch.tensor([[1, 2, 3], [4, 5, 6]], dtype=torch.float32) #我们可以通过传入一个Python列表来创建一个张量。我们还可以指定数据类型dtype。

tensor([[1., 2., 3.],
        [4., 5., 6.]])

## 2.1.2 运算符

按元素（elementwise）运算


In [11]:
x = torch.tensor([1,2,4,8])
y = torch.tensor([2,2,2,2])
x+y, x-y, x*y, x/y, x**y # 张量的加、减、乘、除和幂运算

(tensor([ 3,  4,  6, 10]),
 tensor([-1,  0,  2,  6]),
 tensor([ 2,  4,  8, 16]),
 tensor([0.5000, 1.0000, 2.0000, 4.0000]),
 tensor([ 1,  4, 16, 64]))

In [12]:
torch.exp(x) # 指数运算

tensor([2.7183e+00, 7.3891e+00, 5.4598e+01, 2.9810e+03])

In [13]:
X = torch.arange(12, dtype=torch.float32).reshape((3,4))
Y = torch.tensor([[2.0, 1, 4, 3], [1, 2, 3, 4], [4, 3, 2, 1]])
torch.cat((X, Y), dim=0), torch.cat((X, Y), dim=1) # dim=0表示按行连接，dim=1表示按列连接

(tensor([[ 0.,  1.,  2.,  3.],
         [ 4.,  5.,  6.,  7.],
         [ 8.,  9., 10., 11.],
         [ 2.,  1.,  4.,  3.],
         [ 1.,  2.,  3.,  4.],
         [ 4.,  3.,  2.,  1.]]),
 tensor([[ 0.,  1.,  2.,  3.,  2.,  1.,  4.,  3.],
         [ 4.,  5.,  6.,  7.,  1.,  2.,  3.,  4.],
         [ 8.,  9., 10., 11.,  4.,  3.,  2.,  1.]]))

In [14]:
X == Y # 逐元素比较

tensor([[False,  True, False,  True],
        [False, False, False, False],
        [False, False, False, False]])

In [15]:
X.sum() # 求和

tensor(66.)

### 2.1.3 广播机制（broadcasting mechanism）

工作方式
1. 通过适当复制元素来扩展一个或两个数组，以便在转换之后，两个张量具有相同的形状；
2. 对生成的数组执行按元素操作。

在大多数情况下，我们将沿着数组中长度为1的轴进行广播。

In [16]:
a = torch.arange(3).reshape((3, 1))
b = torch.arange(2).reshape((1, 2))
a, b

(tensor([[0],
         [1],
         [2]]),
 tensor([[0, 1]]))

In [17]:
a+b # 广播机制，矩阵a将复制列， 矩阵b将复制行，然后再按元素相加

tensor([[0, 1],
        [1, 2],
        [2, 3]])

### 2.1.4 索引和切片



索引的写法和 NumPy 一致：下标从 0 开始，`-1` 表示最后一个元素，切片 `a:b` 取第 $a$ 到 $b-1$ 个元素。多维张量用逗号分隔各轴的下标，`:` 表示该轴取全部。

In [18]:
X = torch.arange(12, dtype=torch.float32).reshape(3, 4)
X, X[0], X[-1], X[1:3]      # 第 0 行、最后一行、第 1~2 行

(tensor([[ 0.,  1.,  2.,  3.],
         [ 4.,  5.,  6.,  7.],
         [ 8.,  9., 10., 11.]]),
 tensor([0., 1., 2., 3.]),
 tensor([ 8.,  9., 10., 11.]),
 tensor([[ 4.,  5.,  6.,  7.],
         [ 8.,  9., 10., 11.]]))

In [19]:
X[1, 2] = 9              # 给单个元素赋值
X[0:2, :] = 12           # 前两行整行赋值
X

tensor([[12., 12., 12., 12.],
        [12., 12., 12., 12.],
        [ 8.,  9., 10., 11.]])

### 2.1.5 节省内存

执行运算时，结果一般会写到新分配的内存里。`Y = Y + X` 之后 `id(Y)` 变了，说明 `Y` 指向了另一块内存，原来那块旧张量没人引用，等着被回收。

这在深度学习里不合适：参数可能有几百兆，一秒之内要更新很多次，反复分配内存很快就会耗尽；而且别的变量可能还指着旧内存，读到的就是旧参数。原地更新的写法是把结果写回已经分配好的张量：

$$Y[:] = \mathbf{X} + \mathbf{Y}, \qquad \mathbf{X} \mathrel{+}= \mathbf{Y}.$$

### 2.1.6 转换为其他 Python 对象

`X.numpy()` 把张量转成 NumPy 数组，`torch.tensor(A)` 转回来。两者共享同一块内存，就地修改一个，另一个也跟着变。只有一个元素的张量可以用 `item()`、`float()`、`int()` 取出 Python 标量。

In [20]:
X = torch.arange(12, dtype=torch.float32).reshape(3, 4)
Y = torch.tensor([[2.0, 1, 4, 3], [1, 2, 3, 4], [4, 3, 2, 1]])

before = id(Y)
Y = Y + X
print('Y = Y + X 之后 id 不变:', id(Y) == before)      # False，换了新内存

Z = torch.zeros_like(Y)
print('id(Z):', id(Z))
Z[:] = X + Y
print('id(Z):', id(Z))                                # 不变，原地写入

before = id(X)
X += Y
print('X += Y 之后 id 不变:', id(X) == before)          # True

Y = Y + X 之后 id 不变: False
id(Z): 2418305904592
id(Z): 2418305904592
X += Y 之后 id 不变: True


In [21]:
A = X.numpy()
B = torch.tensor(A)
print(type(A), type(B))

a = torch.tensor([3.5])
a, a.item(), float(a), int(a)      # 单元素张量转 Python 标量

<class 'numpy.ndarray'> <class 'torch.Tensor'>


(tensor([3.5000]), 3.5, 3.5, 3)

### 习题 2.1

1. 运行本节中的代码。将本节中的条件语句X == Y更改为X < Y或X > Y，然后看看你可以得到什么样的张量。

2. 用其他形状（例如三维张量）替换广播机制中按元素操作的两个张量。结果是否与预期相同？

**解：**

1. 把 `X == Y` 换成 `X < Y` 或 `X > Y`，得到的仍是同形状的布尔张量。比较是逐元素做的，形状不变，只是 True 出现的位置不同。

2. 广播不限于二维。规则是从最后一维往前对齐，每一维要么相等，要么其中一个是 1。$(2,3,1)$ 与 $(1,1,4)$ 相加，前两维都有一边是 1、最后一维也有一边是 1，于是广播成 $(2,3,4)$，效果是每个 $3 \times 1$ 的列被加上同一个长度为 4 的向量。

In [22]:
X = torch.arange(12, dtype=torch.float32).reshape(3, 4)
Y = torch.tensor([[2.0, 1, 4, 3], [1, 2, 3, 4], [4, 3, 2, 1]])
X < Y, X > Y

(tensor([[ True, False,  True, False],
         [False, False, False, False],
         [False, False, False, False]]),
 tensor([[False, False, False, False],
         [ True,  True,  True,  True],
         [ True,  True,  True,  True]]))

In [23]:
a = torch.arange(6).reshape((2, 3, 1))
b = torch.arange(4).reshape((1, 1, 4))
print(a.shape, '+', b.shape, '->', (a + b).shape)
a, b, a + b

torch.Size([2, 3, 1]) + torch.Size([1, 1, 4]) -> torch.Size([2, 3, 4])


(tensor([[[0],
          [1],
          [2]],
 
         [[3],
          [4],
          [5]]]),
 tensor([[[0, 1, 2, 3]]]),
 tensor([[[0, 1, 2, 3],
          [1, 2, 3, 4],
          [2, 3, 4, 5]],
 
         [[3, 4, 5, 6],
          [4, 5, 6, 7],
          [5, 6, 7, 8]]]))

In [24]:
a2 = torch.arange(6).reshape((2, 3, 1))
b2 = torch.arange(12).reshape((1, 3, 4))
print(a2.shape, '+', b2.shape, '->', (a2 + b2).shape)
a2, b2, a2 + b2

torch.Size([2, 3, 1]) + torch.Size([1, 3, 4]) -> torch.Size([2, 3, 4])


(tensor([[[0],
          [1],
          [2]],
 
         [[3],
          [4],
          [5]]]),
 tensor([[[ 0,  1,  2,  3],
          [ 4,  5,  6,  7],
          [ 8,  9, 10, 11]]]),
 tensor([[[ 0,  1,  2,  3],
          [ 5,  6,  7,  8],
          [10, 11, 12, 13]],
 
         [[ 3,  4,  5,  6],
          [ 8,  9, 10, 11],
          [13, 14, 15, 16]]]))

## 2.2 数据预处理

实际数据一般存在 CSV 或表格文件里，中间夹着缺失值和类别列。要用它训练模型，得先读进来、补上缺失值、把类别转成数值，最后变成张量。

下面手工造一个四行三列的小数据集：房间数（NumRooms）、巷子类型（Alley）、房价（Price），其中若干项写成 `NA`。



### 2.2.1 处理缺失值

`NaN` 表示缺失。常用的两类办法是插值法和删除法：插值用一个替代值补上，删除则直接丢掉含缺失的行或列。这里用插值。

`iloc` 按位置下标取列，前两列是输入 `inputs`，最后一列是标签 `outputs`。数值列的缺失用同列均值填上；类别列（`Alley` 只有 `Pave` 和缺失两种取值）把缺失当成一个单独的类别，`pd.get_dummies(..., dummy_na=True)` 会把它拆成 `Alley_Pave` 和 `Alley_nan` 两列，取值 1/0。

### 2.2.2 转换为张量格式

处理完以后所有条目都是数值（拆出来的布尔列也能转成 0.0/1.0），用 `to_numpy(dtype=float)` 转成 NumPy 数组，再用 `torch.tensor` 变成张量，后面就能用 2.1 里的那些函数。

In [25]:
import os
import pandas as pd

os.makedirs('data', exist_ok=True)
data_file = os.path.join('data', 'house_tiny.csv')
with open(data_file, 'w') as f:
    f.write('NumRooms,Alley,Price\n')   # 列名
    f.write('NA,Pave,127500\n')         # 每行是一个样本
    f.write('2,NA,106000\n')
    f.write('4,NA,178100\n')
    f.write('NA,NA,140000\n')

data = pd.read_csv(data_file)
data

,NumRooms,Alley,Price
0,NaN,Pave,127500
1,2.0,NaN,106000
2,4.0,NaN,178100
3,NaN,NaN,140000


In [26]:
inputs, outputs = data.iloc[:, 0:2], data.iloc[:, 2]
inputs = inputs.fillna(inputs.mean(numeric_only=True))   # 数值列用均值插值
print(inputs)
inputs = pd.get_dummies(inputs, dummy_na=True)           # 类别列把 NaN 单列一类
print(inputs)

   NumRooms Alley
0       3.0  Pave
1       2.0   NaN
2       4.0   NaN
3       3.0   NaN
   NumRooms  Alley_Pave  Alley_nan
0       3.0        True      False
1       2.0       False       True
2       4.0       False       True
3       3.0       False       True


In [27]:
X = torch.tensor(inputs.to_numpy(dtype=float))
y = torch.tensor(outputs.to_numpy(dtype=float))
X, y

(tensor([[3., 1., 0.],
         [2., 0., 1.],
         [4., 0., 1.],
         [3., 0., 1.]], dtype=torch.float64),
 tensor([127500., 106000., 178100., 140000.], dtype=torch.float64))

### 习题 2.2

创建包含更多行和列的原始数据集。

1. 删除缺失值最多的列。

2. 将预处理后的数据集转换为张量格式。

**解：**

1. 缺失最多的列可以用 `isna().sum()` 数一下，再用 `idxmax()` 取最大值所在的列名。

2. 删完之后，用 `fillna(df.mean(numeric_only=True))` 补均值

In [28]:
import os
import pandas as pd
import torch


#  创建包含更多行和列的原始数据集
os.makedirs(os.path.join('..', 'data'), exist_ok=True)
data_file = os.path.join('..', 'data', 'house_tiny.csv')

with open(data_file, 'w') as f:
    f.write('NumRooms,Alley,Price,NumBathrooms,Street\n')
    f.write('NA,Pave,127500,2,Pave\n')
    f.write('2,NA,106000,1,NA\n')
    f.write('4,NA,178100,3,Pave\n')
    f.write('NA,NA,140000,NA,NA\n')
    f.write('3,Pave,150000,2,Pave\n')
    f.write('5,NA,160000,NA,NA\n')

data = pd.read_csv(data_file)
print("原始数据：")
print(data)
print()


#  删除缺失值最多的列
missing_counts = data.isna().sum()
print("每列缺失值数量：")
print(missing_counts)
print()

max_missing_col = missing_counts.idxmax()
print(f"缺失值最多的列是：{max_missing_col}"
      f"（{missing_counts[max_missing_col]} 个缺失值）")

data = data.drop(columns=[max_missing_col])
print("\n删除该列后的数据：")
print(data)
print()

# 处理剩余缺失值，并转换为张量格式
numeric_cols = data.select_dtypes(include='number').columns
data[numeric_cols] = data[numeric_cols].fillna(data[numeric_cols].mean())

# 剩余类别列：独热编码（NA 行默认编码为全 0）
data = pd.get_dummies(data, dummy_na=True)   
print(data)
# 转成张量
tensor_data = torch.tensor(data.to_numpy(dtype=float))
print("\n转换为张量后的形状：", tensor_data.shape)
print(tensor_data)

原始数据：
   NumRooms Alley   Price  NumBathrooms Street
0       NaN  Pave  127500           2.0   Pave
1       2.0   NaN  106000           1.0    NaN
2       4.0   NaN  178100           3.0   Pave
3       NaN   NaN  140000           NaN    NaN
4       3.0  Pave  150000           2.0   Pave
5       5.0   NaN  160000           NaN    NaN

每列缺失值数量：
NumRooms        2
Alley           4
Price           0
NumBathrooms    2
Street          3
dtype: int64

缺失值最多的列是：Alley（4 个缺失值）

删除该列后的数据：
   NumRooms   Price  NumBathrooms Street
0       NaN  127500           2.0   Pave
1       2.0  106000           1.0    NaN
2       4.0  178100           3.0   Pave
3       NaN  140000           NaN    NaN
4       3.0  150000           2.0   Pave
5       5.0  160000           NaN    NaN

   NumRooms   Price  NumBathrooms  Street_Pave  Street_nan
0       3.5  127500           2.0         True       False
1       2.0  106000           1.0        False        True
2       4.0  178100           3.0         True      

## 2.3 线性代数

### 2.3.1 标量、向量、矩阵与张量

标量是单个数值，用一个元素的张量表示。$x \in \mathbb{R}$ 表示 $x$ 是实值标量。

向量是一列标量值，$\mathbf{x} \in \mathbb{R}^n$：

$$\mathbf{x} = \begin{bmatrix} x_1 \\ x_2 \\ \vdots \\ x_n \end{bmatrix},$$

第 $i$ 个分量记作 $x_i$。代码里对应一维张量，它的"维度"指分量个数 $n$，`len(x)` 和 `x.shape[0]` 都是这个数。

矩阵是二维张量，$\mathbf{A} \in \mathbb{R}^{m \times n}$，元素 $a_{ij}$ 位于第 $i$ 行第 $j$ 列，形状为 $(m, n)$；行数等于列数时叫方阵。转置交换行列，$\mathbf{B} = \mathbf{A}^\top$ 满足 $b_{ij} = a_{ji}$，形状变成 $n \times m$：

$$\mathbf{A}^\top = \begin{bmatrix} a_{11} & a_{21} & \cdots & a_{m1} \\ a_{12} & a_{22} & \cdots & a_{m2} \\ \vdots & \vdots & \ddots & \vdots \\ a_{1n} & a_{2n} & \cdots & a_{mn} \end{bmatrix}.$$

方阵若满足 $\mathbf{A} = \mathbf{A}^\top$ 就叫对称矩阵。

再往上加轴就是更高阶的张量，图像一般写成（通道，高度，宽度）的三阶张量。注意"张量的维度"指轴的个数，"某个轴的维数"指该轴的长度，读文献时这两个说法经常混用。

In [29]:
x = torch.tensor(3.0)     # 标量
y = torch.tensor(2.0)
x + y, x * y, x / y, x ** y

(tensor(5.), tensor(6.), tensor(1.5000), tensor(9.))

In [30]:
x = torch.arange(4)                  # 向量
print(x[3], len(x), x.shape)         # 3 4 torch.Size([4])

tensor(3) 4 torch.Size([4])


In [31]:
A = torch.arange(20).reshape(5, 4)   # 矩阵
B = torch.tensor([[1, 2, 3], [2, 0, 4], [3, 4, 5]])
print(A.shape, '->', A.T.shape)
print('B 是对称矩阵:', (B == B.T).all().item())

torch.Size([5, 4]) -> torch.Size([4, 5])
B 是对称矩阵: True


In [32]:
X = torch.arange(24).reshape(2, 3, 4)   # 三阶张量
X

tensor([[[ 0,  1,  2,  3],
         [ 4,  5,  6,  7],
         [ 8,  9, 10, 11]],

        [[12, 13, 14, 15],
         [16, 17, 18, 19],
         [20, 21, 22, 23]]])

### 2.3.2 按元素运算

形状相同的两个张量做按元素运算，结果形状不变；张量与标量运算时标量作用到每个元素上，形状同样不变。把标量函数搬到向量上就是按元素运算，$\mathbf{c} = F(\mathbf{u}, \mathbf{v})$ 中 $c_i = f(u_i, v_i)$。

一个一元运算不改变操作数的形状。两个矩阵的按元素乘法叫 Hadamard 积，记作 $\mathbf{A} \odot \mathbf{B}$：

$$(\mathbf{A} \odot \mathbf{B})_{ij} = a_{ij} b_{ij},$$

它和矩阵乘法 $\mathbf{A}\mathbf{B}$ 完全是两回事。逻辑运算 `X == Y`、`X < Y` 也是逐元素做的，结果是与操作数同形状的布尔张量。

### 2.3.3 降维

求和会把张量压成一个标量：

$$\sum_{i=1}^{m}\sum_{j=1}^{n} a_{ij}.$$

指定轴求和时，该轴的维数从结果里消失：`axis=0` 得到每列的和、形状 $(n,)$，`axis=1` 得到每行的和、形状 $(m,)$，传一个轴列表则一次压掉多个轴。用 `keepdims=True` 可以保留被压掉的那一维，得到 $(m,1)$，这样就能和原张量直接广播相除，做按行归一化。`cumsum` 求累积和，不改变形状。

均值就是和除以元素个数：

$$\frac{1}{mn}\sum_{i=1}^{m}\sum_{j=1}^{n} a_{ij},$$

`mean` 同样支持按轴计算。

In [33]:
A = torch.arange(20, dtype=torch.float32).reshape(5, 4)
B = A.clone()      # clone 另分配内存，B 与 A 不共享
A + B, A * B       # 按元素加、Hadamard 积

(tensor([[ 0.,  2.,  4.,  6.],
         [ 8., 10., 12., 14.],
         [16., 18., 20., 22.],
         [24., 26., 28., 30.],
         [32., 34., 36., 38.]]),
 tensor([[  0.,   1.,   4.,   9.],
         [ 16.,  25.,  36.,  49.],
         [ 64.,  81., 100., 121.],
         [144., 169., 196., 225.],
         [256., 289., 324., 361.]]))

In [34]:
X = torch.arange(24).reshape(2, 3, 4)
X.shape, (2 + X).shape, (2 * X).shape      # 与标量运算不改变形状

(torch.Size([2, 3, 4]), torch.Size([2, 3, 4]), torch.Size([2, 3, 4]))

In [35]:
print('axis=0:', A.sum(axis=0).shape, ' axis=1:', A.sum(axis=1).shape)
print('沿所有轴求和等于全部求和:', (A.sum(axis=[0, 1]) == A.sum()).item())
A.mean(), A.sum() / A.numel(), A.mean(axis=0)

axis=0: torch.Size([4])  axis=1: torch.Size([5])
沿所有轴求和等于全部求和: True


(tensor(9.5000), tensor(9.5000), tensor([ 8.,  9., 10., 11.]))

In [36]:
sum_A = A.sum(axis=1, keepdims=True)   # 形状 (5,1)
print(sum_A.shape)
print(sum_A)
A / sum_A                              # 广播：每行除以该行之和

torch.Size([5, 1])
tensor([[ 6.],
        [22.],
        [38.],
        [54.],
        [70.]])


tensor([[0.0000, 0.1667, 0.3333, 0.5000],
        [0.1818, 0.2273, 0.2727, 0.3182],
        [0.2105, 0.2368, 0.2632, 0.2895],
        [0.2222, 0.2407, 0.2593, 0.2778],
        [0.2286, 0.2429, 0.2571, 0.2714]])

In [37]:
A.cumsum(axis=0)     # 沿轴 0 的累积和，形状不变

tensor([[ 0.,  1.,  2.,  3.],
        [ 4.,  6.,  8., 10.],
        [12., 15., 18., 21.],
        [24., 28., 32., 36.],
        [40., 45., 50., 55.]])

### 2.3.4 点积、矩阵-向量积与矩阵乘法

两个向量的点积是相同位置乘积之和：

$$\mathbf{x}^\top \mathbf{y} = \sum_{i=1}^{d} x_i y_i .$$

权重非负且和为 1 时，点积就是加权平均；两个向量归一化之后，点积给出夹角的余弦。`torch.dot` 只接受一维张量，等价于 `torch.sum(x * y)`。

把 $\mathbf{A} \in \mathbb{R}^{m \times n}$ 按行写成 $\mathbf{a}_1^\top, \ldots, \mathbf{a}_m^\top$，矩阵-向量积的每个分量是 $\mathbf{A}$ 的一行与 $\mathbf{x}$ 的点积：

$$\mathbf{A}\mathbf{x} = \begin{bmatrix} \mathbf{a}_1^\top \mathbf{x} \\ \mathbf{a}_2^\top \mathbf{x} \\ \vdots \\ \mathbf{a}_m^\top \mathbf{x} \end{bmatrix} \in \mathbb{R}^m .$$

$\mathbf{A} \in \mathbb{R}^{n \times k}$ 与 $\mathbf{B} \in \mathbb{R}^{k \times m}$ 相乘得 $\mathbf{C} = \mathbf{A}\mathbf{B} \in \mathbb{R}^{n \times m}$，其中

$$c_{ij} = \mathbf{a}_i^\top \mathbf{b}_j = \sum_{r=1}^{k} a_{ir} b_{rj} .$$

这相当于做 $m$ 次矩阵-向量积再拼起来，要求前一个的列数等于后一个的行数。矩阵乘法不能写成 $\mathbf{B}\mathbf{A}$，也不等于 Hadamard 积。

In [38]:
x = torch.arange(4, dtype=torch.float32)
y = torch.ones(4, dtype=torch.float32)
torch.dot(x, y), torch.sum(x * y)      # 点积的两种写法

(tensor(6.), tensor(6.))

In [39]:
A = torch.arange(20, dtype=torch.float32).reshape(5, 4)
print(A.shape, x.shape, '->', torch.mv(A, x).shape)
torch.mv(A, x)

torch.Size([5, 4]) torch.Size([4]) -> torch.Size([5])


tensor([ 14.,  38.,  62.,  86., 110.])

In [40]:
B = torch.ones(4, 3)
torch.mm(A, B)      # (5,4) @ (4,3) -> (5,3)

tensor([[ 6.,  6.,  6.],
        [22., 22., 22.],
        [38., 38., 38.],
        [54., 54., 54.],
        [70., 70., 70.]])

### 2.3.5 范数

范数把向量映射成非负标量，用来度量"有多大"。它要满足齐次性 $f(\alpha \mathbf{x}) = |\alpha| f(\mathbf{x})$、三角不等式 $f(\mathbf{x} + \mathbf{y}) \le f(\mathbf{x}) + f(\mathbf{y})$、非负性，并且只有零向量的范数为 0。

最常用的是 $L_2$ 范数（欧氏距离）和 $L_1$ 范数：

$$\|\mathbf{x}\|_2 = \sqrt{\sum_{i=1}^{n} x_i^2}, \qquad \|\mathbf{x}\|_1 = \sum_{i=1}^{n} |x_i| .$$

两者都是 $L_p$ 范数的特例：

$$\|\mathbf{x}\|_p = \left(\sum_{i=1}^{n} |x_i|^p\right)^{1/p} .$$

$L_1$ 受异常值影响小一些，$L_2$ 的平方在求导时更干净，所以深度学习里更常出现 $\|\mathbf{x}\|_2^2$。矩阵的 Frobenius 范数就是把所有元素拉平之后的 $L_2$ 范数：

$$\|\mathbf{X}\|_F = \sqrt{\sum_{i=1}^{m}\sum_{j=1}^{n} x_{ij}^2}.$$

目标函数基本都能写成范数：最小化预测与真实值的距离就是最小化某个范数，L2 正则则是给参数加上 $\|\mathbf{w}\|_2^2$ 一项。新版 PyTorch 用 `torch.linalg.norm`，`torch.norm` 已经不再推荐。

In [41]:
u = torch.tensor([3.0, -4.0])
torch.linalg.norm(u), torch.abs(u).sum()      # L2 范数 5，L1 范数 7

(tensor(5.), tensor(7.))

In [42]:
torch.linalg.norm(torch.ones((4, 9)))      # Frobenius 范数，sqrt(36) = 6

tensor(6.)

### 习题 2.3

1. 证明一个矩阵 $\mathbf{A}$ 的转置的转置是 $\mathbf{A}$，即 $(\mathbf{A}^\top)^\top = \mathbf{A}$。

2. 给出两个矩阵 $\mathbf{A}$ 和 $\mathbf{B}$，证明"它们转置的和"等于"它们和的转置"，即 $\mathbf{A}^\top + \mathbf{B}^\top = (\mathbf{A} + \mathbf{B})^\top$。

3. 给定任意方阵 $\mathbf{A}$，$\mathbf{A} + \mathbf{A}^\top$ 总是对称的吗？为什么？

4. 本节中定义了形状 $(2,3,4)$ 的张量 `X`。`len(X)` 的输出结果是什么？

5. 对于任意形状的张量 `X`，`len(X)` 是否总是对应于 `X` 特定轴的长度？这个轴是什么？

6. 运行 `A / A.sum(axis=1)`，看看会发生什么。请分析一下原因。

7. 考虑一个具有形状 $(2,3,4)$ 的张量，在轴 0、1、2 上的求和输出是什么形状？

8. 为 `linalg.norm` 函数提供 3 个或更多轴的张量，并观察其输出。对于任意形状的张量，这个函数计算得到什么？

**解：**

1. $(\mathbf{A}^\top)^\top = \mathbf{A}$。转置只是交换下标，$(\mathbf{A}^\top)_{ij} = a_{ji}$，再转置一次又回到 $a_{ij}$，逐元素相等。

2. $\mathbf{A}^\top + \mathbf{B}^\top = (\mathbf{A} + \mathbf{B})^\top$。左边第 $(i,j)$ 项是 $a_{ji} + b_{ji}$，右边也是这个值，所以两个矩阵相等。

3. 一定对称。$(\mathbf{A} + \mathbf{A}^\top)^\top = \mathbf{A}^\top + (\mathbf{A}^\top)^\top = \mathbf{A}^\top + \mathbf{A} = \mathbf{A} + \mathbf{A}^\top$，它等于自己的转置。

4. `X` 形状为 $(2,3,4)$ 时，`len(X)` 是 2。

5. `len(X)` 总是轴 0 的长度，因为一级索引走的就是最外层。

6. `A / A.sum(axis=1)` 会报错。`A` 是 $(5,4)$，`A.sum(axis=1)` 是 $(5,)$；广播时从最后一维往前对齐，4 与 5 既不相等也没有 1，无法扩展。加 `keepdims=True` 变成 $(5,1)$ 就正常了。

7. 形状 $(2,3,4)$ 的张量沿轴 0、1、2 求和，结果形状分别是 $(3,4)$、$(2,4)$、$(2,3)$。

8. 对三维以上的张量，`torch.linalg.norm` 不给 `dim` 时会把所有元素当成一个向量求 $L_2$ 范数，输出一个标量，数值上等于 $\sqrt{\sum_{ijk} x_{ijk}^2}$。

In [43]:
A = torch.arange(20, dtype=torch.float32).reshape(5, 4)
B = torch.randn(5, 4)
print('1:', (A.T.T == A).all().item())
print('2:', (A.T + B.T == (A + B).T).all().item())

C = torch.randn(4, 4)
S = C + C.T
print('3:', (S == S.T).all().item())

1: True
2: True
3: True


In [44]:
X = torch.arange(24, dtype=torch.float32).reshape(2, 3, 4)
print('4: len(X) =', len(X))
print('7:', X.sum(axis=0).shape, X.sum(axis=1).shape, X.sum(axis=2).shape)
print('8:', torch.linalg.norm(X).item(), X.pow(2).sum().sqrt().item())

4: len(X) = 2
7: torch.Size([3, 4]) torch.Size([2, 4]) torch.Size([2, 3])
8: 65.75712585449219 65.75712585449219


In [45]:
try:
    A / A.sum(axis=1)          # (5,4) / (5,)：最后一维 4 与 5 无法对齐
except RuntimeError as e:
    print('6: RuntimeError:', e)

(A / A.sum(axis=1, keepdims=True)).shape    # 改成 (5,1) 就能广播

6: RuntimeError: The size of tensor a (4) must match the size of tensor b (5) at non-singleton dimension 1


torch.Size([5, 4])

## 2.4 微积分

### 2.4.1 导数与微分

训练模型就是让损失变小，所以需要知道损失对每个参数的变化率。设 $f: \mathbb{R} \to \mathbb{R}$，导数定义为

$$f'(x) = \lim_{h \rightarrow 0} \frac{f(x+h) - f(x)}{h},$$

它是 $f$ 在 $x$ 处的瞬时变化率，也等于曲线在该点切线的斜率。$f'(a)$ 存在就说 $f$ 在 $a$ 处可微。

拿 $u = f(x) = 3x^2 - 4x$ 试一下。令 $x = 1$，让 $h$ 依次缩小十倍，$\frac{f(x+h)-f(x)}{h}$ 会向 2 靠拢；用前面的法则逐项求导也是这个结果：

$$u' = 3 \cdot 2x - 4 = 6x - 4 \Big|_{x=1} = 2 .$$

常用的微分规则有 $DC = 0$、$Dx^n = nx^{n-1}$、$De^x = e^x$、$D\ln x = 1/x$，再配合常数倍、加法、乘法、除法法则，多项式就能逐项求导。

### 2.4.2 偏导数、梯度与链式法则

多变量函数对第 $i$ 个变量求偏导，就是把其他变量当成常数：

$$\frac{\partial y}{\partial x_i} = \lim_{h \rightarrow 0} \frac{f(x_1, \ldots, x_i + h, \ldots, x_n) - f(x_1, \ldots, x_n)}{h}.$$

把各偏导拼成列向量就是梯度：

$$\nabla_{\mathbf{x}} f(\mathbf{x}) = \left[\frac{\partial f(\mathbf{x})}{\partial x_1}, \frac{\partial f(\mathbf{x})}{\partial x_2}, \ldots, \frac{\partial f(\mathbf{x})}{\partial x_n}\right]^\top .$$

几个常用结论：$\nabla_{\mathbf{x}} \mathbf{A}\mathbf{x} = \mathbf{A}^\top$，$\nabla_{\mathbf{x}} \mathbf{x}^\top \mathbf{A} = \mathbf{A}$，$\nabla_{\mathbf{x}} \mathbf{x}^\top \mathbf{A} \mathbf{x} = (\mathbf{A} + \mathbf{A}^\top)\mathbf{x}$，$\nabla_{\mathbf{x}} \|\mathbf{x}\|^2 = 2\mathbf{x}$，$\nabla_{\mathbf{X}} \|\mathbf{X}\|_F^2 = 2\mathbf{X}$。梯度指向函数上升最快的方向，所以优化时沿 $-\nabla$ 走一步。

复合函数用链式法则。单变量时 $\frac{dy}{dx} = \frac{dy}{du}\frac{du}{dx}$；有多个中间变量 $u_1, \ldots, u_m$ 时

$$\frac{\partial y}{\partial x_i} = \sum_{j=1}^{m} \frac{\partial y}{\partial u_j}\frac{\partial u_j}{\partial x_i}.$$

神经网络就是一层套一层的复合函数，反向传播时把各层的局部导数沿链乘起来，用的正是这条式子。

In [46]:
import numpy as np

def f(x):
    return 3 * x ** 2 - 4 * x

def numerical_lim(f, x, h):
    return (f(x + h) - f(x)) / h

h = 0.1
for i in range(5):
    print(f'h={h:.5f}, numerical limit={numerical_lim(f, 1, h):.5f}')
    h *= 0.1

h=0.10000, numerical limit=2.30000
h=0.01000, numerical limit=2.03000
h=0.00100, numerical limit=2.00300
h=0.00010, numerical limit=2.00030
h=0.00001, numerical limit=2.00003


In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

x = np.arange(0, 3, 0.1)
plt.figure(figsize=(3.5, 2.5))
plt.plot(x, f(x), label='f(x)')
plt.plot(x, 2 * x - 3, 'm--', label='tangent, x=1')
plt.xlabel('x'); plt.ylabel('f(x)')
plt.legend(); plt.grid(); plt.show()


C:\Users\LU\AppData\Local\Temp\ipykernel_10124\2454619661.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.legend(); plt.grid(); plt.show()


### 习题 2.4

1. 绘制函数 $y = f(x) = x^3 - \frac{1}{x}$ 和其在 $x = 1$ 处切线的图像。

2. 求函数 $f(\mathbf{x}) = 3x_1^2 + 5e^{x_2}$ 的梯度。

3. 函数 $f(\mathbf{x}) = \|\mathbf{x}\|_2$ 的梯度是什么？

4. 尝试写出函数 $u = f(x, y, z)$，其中 $x = x(a, b)$、$y = y(a, b)$、$z = z(a, b)$ 的链式法则。

**解：**

1. $y = x^3 - \frac{1}{x}$，导数是 $y' = 3x^2 + \frac{1}{x^2}$。$x = 1$ 时 $y = 0$、$y' = 4$，切线为 $y = 4(x - 1) = 4x - 4$。

2. $f(\mathbf{x}) = 3x_1^2 + 5e^{x_2}$，对两个变量分别求偏导得 $\nabla f = [6x_1,\ 5e^{x_2}]^\top$。

3. $\|\mathbf{x}\|_2 = (\mathbf{x}^\top \mathbf{x})^{1/2}$，用链式法则得

$$\nabla_{\mathbf{x}} \|\mathbf{x}\|_2 = \frac{\mathbf{x}}{\|\mathbf{x}\|_2} \quad (\mathbf{x} \ne \mathbf{0}),$$

也就是同方向的单位向量；原点处不可微。

4. $x = x(a,b)$、$y = y(a,b)$、$z = z(a,b)$、$u = f(x,y,z)$ 时

$$\frac{\partial u}{\partial a} = \frac{\partial u}{\partial x}\frac{\partial x}{\partial a} + \frac{\partial u}{\partial y}\frac{\partial y}{\partial a} + \frac{\partial u}{\partial z}\frac{\partial z}{\partial a},$$

对 $b$ 的式子把 $\partial a$ 换成 $\partial b$ 即可。三条中间路径各贡献一项，正好体现"沿所有路径求和"。

In [48]:
g = lambda t: t ** 3 - 1 / t

xs = np.arange(0.1, 3, 0.1)
plt.figure(figsize=(3.5, 2.5))
plt.plot(xs, g(xs), label='g(x) = x^3 - 1/x')
plt.plot(xs, 4 * xs - 4, 'm--', label='tangent, x=1')
plt.ylim(-6, 10); plt.legend(); plt.grid(); plt.show()

print('数值逼近:', numerical_lim(g, 1, 1e-6), ' 解析值:', 3 * 1 ** 2 + 1 / 1 ** 2)

数值逼近: 4.000001999737712  解析值: 4.0


C:\Users\LU\AppData\Local\Temp\ipykernel_10124\3565627029.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.ylim(-6, 10); plt.legend(); plt.grid(); plt.show()


In [49]:
# 用自动微分核对第 2、3 题
x = torch.tensor([1.0, 2.0], requires_grad=True)
(3 * x[0] ** 2 + 5 * torch.exp(x[1])).backward()
print('第 2 题梯度:', x.grad, ' 期望:', torch.tensor([6.0, 5 * np.exp(2)]))

v = torch.tensor([3.0, 4.0], requires_grad=True)
v.norm().backward()
print('第 3 题梯度:', v.grad, ' 期望:', v.detach() / v.detach().norm())

第 2 题梯度: tensor([ 6.0000, 36.9453])  期望: tensor([ 6.0000, 36.9453], dtype=torch.float64)
第 3 题梯度: tensor([0.6000, 0.8000])  期望: tensor([0.6000, 0.8000])


## 2.5 自动微分

手工求导对深层网络不现实。框架的做法是记录运算过程，得到计算图，再反向走一遍链式法则，把输出对每个输入的偏导填进去，这就是反向传播。

以 $y = 2\mathbf{x}^\top\mathbf{x}$ 关于 $\mathbf{x}$ 求导为例：先让 `x` 记录运算，再算 `y`，最后调用 `backward()`，梯度就写到 `x.grad` 里，形状与 `x` 相同。理论结果是

$$\frac{\partial y}{\partial \mathbf{x}} = 4\mathbf{x}.$$

PyTorch 是累加梯度的，同一个 `x` 反传两次梯度会叠加，所以第二次用之前要 `x.grad.zero_()`。

### 2.5.1 非标量输出

`y` 不是标量时，`backward()` 需要传一个与 `y` 同形状的梯度参数，表示损失对 `y` 的导数。训练时通常对 batch 里所有样本的损失求和再反传，即

$$\frac{\partial}{\partial \mathbf{x}} \sum_{i} y_i ,$$

所以传全 1 的向量就够了，等价于先 `y.sum()` 再 `backward()`。

### 2.5.2 分离计算

`y.detach()` 返回数值相同、但不带计算历史的张量。用它算 $z = u x$ 时，梯度不会沿着 $u$ 传回 $\mathbf{x}$，于是 $\partial z / \partial \mathbf{x} = u$，而不是 $3x^2$。这种做法常用来把某个量当作常数，比如固定住旧模型的输出去做目标值。

### 2.5.3 控制流

计算图是运行时按实际走的分支建立的，所以带 `if`、`while`、函数调用的代码同样能求导。下面 `f` 里 $b$ 反复乘 2 直到范数超过 1000，再按符号决定要不要乘 100，整体仍是分段线性的 $f(a) = ka$，所以 `a.grad` 应该等于 `d / a`。

In [50]:
x = torch.arange(4.0)
x.requires_grad_(True)        # 等价于 torch.arange(4.0, requires_grad=True)
print('反传前 x.grad =', x.grad)

y = 2 * torch.dot(x, x)
y.backward()
x.grad, x.grad == 4 * x

反传前 x.grad = None


(tensor([ 0.,  4.,  8., 12.]), tensor([True, True, True, True]))

In [51]:
x.grad.zero_()      # 梯度会累加，先清零
y = x.sum()
y.backward()
x.grad

tensor([1., 1., 1., 1.])

In [52]:
x.grad.zero_()
y = x * x
y.sum().backward()      # 等价于 y.backward(torch.ones(len(x)))
x.grad

tensor([0., 2., 4., 6.])

In [53]:
x.grad.zero_()
y = x * x
u = y.detach()          # 数值同 y，但没有计算历史
z = u * x
z.sum().backward()
print('dz/dx == u:', x.grad == u)

x.grad.zero_()
y.sum().backward()      # 对 y 本身反传，得到 2x
print('dy/dx == 2x:', x.grad == 2 * x)

dz/dx == u: tensor([True, True, True, True])
dy/dx == 2x: tensor([True, True, True, True])


In [54]:
def f_flow(a):
    b = a * 2
    while b.norm() < 1000:      # 迭代次数取决于 a
        b = b * 2
    if b.sum() > 0:             # 分支也取决于 a
        c = b
    else:
        c = 100 * b
    return c

a = torch.randn(size=(), requires_grad=True)
d = f_flow(a)
d.backward()
a, d, a.grad, a.grad == d / a

(tensor(1.9792, requires_grad=True),
 tensor(1013.3304, grad_fn=<MulBackward0>),
 tensor(512.),
 tensor(True))

### 习题 2.5

1. 为什么计算二阶导数比一阶导数的开销要更大？

2. 在运行反向传播函数之后，立即再次运行它，看看会发生什么。

3. 在控制流的例子中，我们计算 `d` 关于 `a` 的导数，如果将变量 `a` 更改为随机向量或矩阵，会发生什么？

4. 重新设计一个求控制流梯度的例子，运行并分析结果。

5. 使 $f(x) = \sin(x)$，绘制 $f(x)$ 和 $\frac{df(x)}{dx}$ 的图像，其中后者不使用 $f'(x) = \cos(x)$。

**解：**

1. 二阶导数开销更大。一阶只要反传一次；求二阶要对梯度再建一层计算图（`create_graph=True`），图更长，中间结果也不能马上释放，时间和内存都会明显上升，所以框架默认只算一阶。

2. 反传之后再调用一次 `backward()` 会报 `Trying to backward through the graph a second time`。原因是反传结束就释放了图中保存的中间结果，这么做是为了省内存；确实需要多次反传时传 `retain_graph=True`。

3. `a` 换成向量或矩阵后，`d` 也是向量或矩阵，`d.backward()` 会报 `grad can be implicitly created only for scalar outputs`。改成 `d.sum().backward()` 即可，此时每个分量各有自己的系数 $k_i$，`a.grad == d / a` 仍然成立。

4. 自己写一个分段函数，例如 $a \ge 0$ 时 $f(a) = 2a$、$a < 0$ 时 $f(a) = 3a$，反传后负元素的梯度是 3，其余是 2，说明框架自动走了各自的分支。

5. 用中心差分 $\frac{\sin(x+h) - \sin(x-h)}{2h}$ 算出的曲线与 $\cos x$ 重合，数值求导可用，但精度受 $h$ 影响：$h$ 太大是截断误差，太小是舍入误差。

In [55]:
# 2. 反传之后再反传一次
a = torch.tensor([1.0, 2.0], requires_grad=True)
b = (a.exp() * a).sum()
b.backward()
try:
    b.backward()
except RuntimeError as e:
    print('RuntimeError:', str(e)[:100], '...')

# 想在同一张图上反传两次，要传 retain_graph=True
a.grad.zero_()
b2 = (a.exp() * a).sum()
b2.backward(retain_graph=True)
b2.backward()
print('两次反传后 a.grad =', a.grad)      # 梯度累加，是单次的两倍

RuntimeError: Trying to backward through the graph a second time (or directly access saved tensors after they have ...
两次反传后 a.grad = tensor([10.8731, 44.3343])


In [56]:
# 3. 输入换成向量
a = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
d = f_flow(a)
print('d =', d)
try:
    d.backward()
except RuntimeError as e:
    print('RuntimeError:', str(e)[:100], '...')

d.sum().backward()                        # 与之前求和再反传等价
print('a.grad == d / a:', (a.grad == d.detach() / a).all().item())

d = tensor([ 512., 1024., 1536.], grad_fn=<MulBackward0>)
RuntimeError: grad can be implicitly created only for scalar outputs ...
a.grad == d / a: True


In [57]:
# 4. 自己设计的分段函数：a>=0 时 2a，a<0 时 3a
def f_piece(a):
    return torch.where(a >= 0, 2 * a, 3 * a)

a = torch.tensor([-1.0, 0.5, 2.0], requires_grad=True)
f_piece(a).sum().backward()
a.grad      # [-1 处是 3，其余是 2]

tensor([3., 2., 2.])

In [58]:
# 5. sin 的数值导数，不用 cos 公式
xs = np.linspace(-2 * np.pi, 2 * np.pi, 400)
dys = np.gradient(np.sin(xs), xs)      # 差分

plt.figure(figsize=(4, 2.5))
plt.plot(xs, np.sin(xs), label='sin(x)')
plt.plot(xs, dys, 'm--', label='numerical derivative')
plt.plot(xs, np.cos(xs), 'g:', label='cos(x)')
plt.legend(); plt.grid(); plt.show()

C:\Users\LU\AppData\Local\Temp\ipykernel_10124\3492590622.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.legend(); plt.grid(); plt.show()


## 2.6 概率

模型输出的是概率，损失函数也多半用概率的语言写（交叉熵、最大似然）。这一节要抓住的几条是：怎么定义概率、怎么用抽样估计它、联合与条件概率、贝叶斯公式、独立性，以及期望和方差。

### 2.6.1 抽样与大数定律

公平骰子每个面概率是 $\frac{1}{6}$。但手上的骰子有没有偏，只能靠试验判断：把某个结果出现的次数除以总投掷次数，投得越多，这个比值越接近真实概率，这就是大数定律。从分布里取样本叫抽样。

`torch.distributions.multinomial` 做多项分布抽样。它的输出是一个与概率向量等长的向量，第 $i$ 项是结果 $i$ 出现的次数；参数 `total_count` 是一次抽多少个样本，`sample((m,))` 一次抽 $m$ 组。

### 2.6.2 概率公理与随机变量

样本空间 $\mathcal{S}$ 是所有可能结果的集合，事件是它的子集，概率 $P$ 满足三条：$P(\mathcal{A}) \ge 0$；$P(\mathcal{S}) = 1$；互斥事件满足 $P(\bigcup_i \mathcal{A}_i) = \sum_i P(\mathcal{A}_i)$。

随机变量是把结果映射到数值的变量，$P(X = a)$ 表示它取值为 $a$ 的概率。离散变量谈概率，连续变量谈密度：身高恰好 1.80 米的概率是 0，但落在某个区间内的概率不是 0。

### 2.6.3 联合概率、条件概率与贝叶斯定理

$P(A = a, B = b)$ 是联合概率。条件概率定义为比率

$$P(B = b \mid A = a) = \frac{P(A = a, B = b)}{P(A = a)},$$

由此得到乘法法则 $P(A, B) = P(B \mid A)P(A)$ 和贝叶斯定理

$$P(A \mid B) = \frac{P(B \mid A)P(A)}{P(B)}.$$

分母靠边际化得到，也就是把 $A$ 的所有取值加起来：

$$P(B) = \sum_{a} P(A = a, B = b) = \sum_{a} P(B = b \mid A = a)P(A = a).$$

$A$ 与 $B$ 独立当且仅当 $P(A, B) = P(A)P(B)$，也等价于 $P(A \mid B) = P(A)$。给定 $C$ 时条件独立写成 $P(A, B \mid C) = P(A \mid C)P(B \mid C)$，两次检查结果互不影响时用这个假设。

### 2.6.4 期望和方差

$$E[X] = \sum_{x} x\, P(X = x), \qquad E[f(X)] = \sum_{x} f(x)\, P(X = x).$$

方差衡量偏离期望的程度：

$$\mathrm{Var}[X] = E\left[(X - E[X])^2\right] = E[X^2] - E[X]^2,$$

开根号得到标准差。

In [59]:
from torch.distributions import multinomial

fair_probs = torch.ones([6]) / 6
print(multinomial.Multinomial(1, fair_probs).sample())                 # 掷一次
print(multinomial.Multinomial(1000, fair_probs).sample() / 1000)       # 掷 1000 次的相对频率

tensor([0., 0., 0., 0., 0., 1.])
tensor([0.1680, 0.1540, 0.1810, 0.1700, 0.1730, 0.1540])


In [60]:
counts = multinomial.Multinomial(10, fair_probs).sample((500,))        # 500 组，每组 10 次
cum_counts = counts.cumsum(dim=0)
estimates = cum_counts / cum_counts.sum(dim=1, keepdims=True)          # keepdims 之后可以广播

plt.figure(figsize=(5, 3.5))
for i in range(6):
    plt.plot(estimates[:, i].numpy(), label=f'P(die={i + 1})')
plt.axhline(y=1 / 6, color='black', linestyle='dashed')
plt.xlabel('Groups of experiments'); plt.ylabel('Estimated probability')
plt.legend(); plt.grid(); plt.show()

C:\Users\LU\AppData\Local\Temp\ipykernel_10124\4250969613.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.legend(); plt.grid(); plt.show()


In [61]:
# 医学检查：罕见病 P(H=1)=0.0015，两次检查在给定 H 时条件独立
P_H1 = 0.0015
P_D1_H1, P_D1_H0 = 0.99, 0.01
P_D2_H1, P_D2_H0 = 0.98, 0.03

P_D1 = P_D1_H1 * P_H1 + P_D1_H0 * (1 - P_H1)              # 边际化
print('P(H=1|D1=1) =', round(P_D1_H1 * P_H1 / P_D1, 4))

P_both_H1 = P_D1_H1 * P_D2_H1 * P_H1                      # 条件独立，概率相乘
P_both_H0 = P_D1_H0 * P_D2_H0 * (1 - P_H1)
print('P(H=1|D1=1,D2=1) =', round(P_both_H1 / (P_both_H1 + P_both_H0), 4))

P(H=1|D1=1) = 0.1295
P(H=1|D1=1,D2=1) = 0.8293


### 习题 2.6

1. 进行 $m = 500$ 组实验，每组抽取 $n = 10$ 个样本。改变 $m$ 和 $n$，观察和分析实验结果。

2. 给定两个概率为 $P(\mathcal{A})$ 和 $P(\mathcal{B})$ 的事件，计算 $P(\mathcal{A} \cup \mathcal{B})$ 和 $P(\mathcal{A} \cap \mathcal{B})$ 的上限和下限。（提示：用韦恩图来展示这些情况。）

3. 假设有一系列随机变量 $A$、$B$ 和 $C$，其中 $B$ 只依赖于 $A$，而 $C$ 只依赖于 $B$，能简化联合概率 $P(A, B, C)$ 吗？（提示：这是一个马尔可夫链。）

4. 在 HIV 检测的例子中，第一个测试更准确。为什么不把第一个测试做两次，而是同时做第一个和第二个测试？

**解：**

1. $m$ 组、每组 $n$ 个样本时，估计精度主要由总样本数 $mn$ 决定。$m$ 越大曲线越平滑；$n$ 很小时每组估计只能取 $0, \frac{1}{n}, \frac{2}{n}, \ldots$ 这些值，起步阶段抖动明显，但样本累积之后同样收敛到 $\frac{1}{6}$。

2. 由 $P(A \cup B) = P(A) + P(B) - P(A \cap B)$ 立即可得交集的上下界

$$\max\left(0,\ P(A) + P(B) - 1\right) \le P(A \cap B) \le \min\left(P(A),\ P(B)\right),$$

并集的上下界则是

$$\max\left(P(A),\ P(B)\right) \le P(A \cup B) \le \min\left(1,\ P(A) + P(B)\right).$$

画韦恩图就能看出四个界各自在什么位置取到：交集最小是两集合尽量分开，交集最大是一个集合被另一个包含。

3. 能简化。$C$ 只依赖 $B$、$B$ 只依赖 $A$，所以给定 $B$ 后 $C$ 与 $A$ 条件独立，联合概率按链式分解成

$$P(A, B, C) = P(A)P(B \mid A)P(C \mid B),$$

这正是马尔可夫链的分解形式。

4. 同一个测试重复做两次，两次结果在同一患者身上高度相关，并不是给定患病状态下的条件独立，重复并不能带来新的证据；而第二个测试的假阳性、假阴性结构与第一个不同，两次一起做能给出互补信息，在同样的检测次数下后验概率提升更明显。

In [62]:
def estimate(m, n):
    """m 组实验、每组 n 次，返回每组结束后的累积频率估计"""
    counts = multinomial.Multinomial(n, fair_probs).sample((m,))
    cum = counts.cumsum(dim=0)
    return (cum / cum.sum(dim=1, keepdims=True)).numpy()

for m, n in [(500, 10), (50, 100), (5000, 1), (10, 500)]:
    est = estimate(m, n)
    print(f'm={m:>4}, n={n:>3}, 总样本 {m * n:>5}: 最后一组估计 =', np.round(est[-1], 3))

m= 500, n= 10, 总样本  5000: 最后一组估计 = [0.166 0.16  0.173 0.169 0.168 0.164]
m=  50, n=100, 总样本  5000: 最后一组估计 = [0.17  0.172 0.168 0.162 0.157 0.171]
m=5000, n=  1, 总样本  5000: 最后一组估计 = [0.169 0.166 0.16  0.172 0.175 0.158]
m=  10, n=500, 总样本  5000: 最后一组估计 = [0.161 0.168 0.173 0.174 0.157 0.167]


In [63]:
def bounds(pA, pB):
    union = (max(pA, pB), min(1.0, pA + pB))
    inter = (max(0.0, pA + pB - 1), min(pA, pB))
    return union, inter

for pA, pB in [(0.3, 0.4), (0.7, 0.8), (0.2, 0.5)]:
    (u_lo, u_hi), (i_lo, i_hi) = bounds(pA, pB)
    print(f'P(A)={pA}, P(B)={pB}: P(A∪B) ∈ [{u_lo:.2f}, {u_hi:.2f}], '
          f'P(A∩B) ∈ [{i_lo:.2f}, {i_hi:.2f}]')

P(A)=0.3, P(B)=0.4: P(A∪B) ∈ [0.40, 0.70], P(A∩B) ∈ [0.00, 0.30]
P(A)=0.7, P(B)=0.8: P(A∪B) ∈ [0.80, 1.00], P(A∩B) ∈ [0.50, 0.70]
P(A)=0.2, P(B)=0.5: P(A∪B) ∈ [0.50, 0.70], P(A∩B) ∈ [0.00, 0.20]


## 2.7 查阅文档

API 很多，记不完，要会用文档。`dir(模块)` 列出模块里的名字，`help(函数)` 打印文档字符串；Jupyter 里还可以在标识符后面加 `?` 看文档、加 `??` 看源码。官网的 API 文档和示例也是必查的。

`help(torch.ones)` 会说明第一个参数是形状，既可以传元组也可以传可变参数。

### 习题 2.7

在深度学习框架中查找任何函数或类的文档，并尝试在这个框架的官方网站上找到它。

**解：**

以 `torch.nn.functional.softmax` 为例，文档里写明它沿 `dim` 指定的轴做归一化：`dim=1` 时每一行加起来是 1，`dim=0` 时每一列加起来是 1。`dtype` 指定计算精度，`_stacklevel` 是内部参数，一般不用管。查文档时先看签名，再看参数说明，最后看示例，基本就能确认用法。

In [64]:
print(len(dir(torch)))                                   # torch 里有多少个名字
print([name for name in dir(torch) if 'rand' in name])   # 按关键字筛
help(torch.ones)

1522
['_philox_randint_', 'quasirandom', 'rand', 'rand_like', 'randint', 'randint_like', 'randn', 'randn_like', 'random', 'randperm']
Help on built-in function ones in module torch:

ones(...)
    ones(*size, *, out=None, dtype=None, layout=torch.strided, device=None, requires_grad=False) -> Tensor

    Returns a tensor filled with the scalar value `1`, with the shape defined
    by the variable argument :attr:`size`.

    Args:
        size (int...): a sequence of integers defining the shape of the output tensor.
            Can be a variable number of arguments or a collection like a list or tuple.

    Keyword arguments:
        out (Tensor, optional): the output tensor.
        dtype (:class:`torch.dtype`, optional): the desired data type of returned tensor.
            Default: if ``None``, uses a global default (see :func:`torch.set_default_dtype`).
        layout (:class:`torch.layout`, optional): the desired layout of returned Tensor.
            Default: ``torch.strided``.
   

In [65]:
import torch.nn.functional as F

probs = F.softmax(torch.tensor([[1., 2., 3.], [1., 1., 1.]]), dim=1)
probs, probs.sum(dim=1)      # 每行归一化，行和为 1

(tensor([[0.0900, 0.2447, 0.6652],
         [0.3333, 0.3333, 0.3333]]),
 tensor([1., 1.]))

## 2.8 小结

张量是框架里唯一的数据载体，形状、索引、广播、原地操作、与 NumPy 互转这些日常操作要熟。表格数据先用 pandas 补缺失值、把类别拆成 0/1，再转成张量。线性代数提供整套符号：标量、向量、矩阵、张量，以及降维、点积、矩阵乘法、范数，后面每个模型都写在这些符号里。微积分给出优化的方向，链式法则支撑反向传播；自动微分把求导交给框架，需要留意 `requires_grad`、梯度清零、`detach` 和非标量输出要传梯度这几件事。概率用来描述不确定性，联合、条件、贝叶斯、独立性和期望方差是后面损失函数的基础。